<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 La Biblioteca del Pueblo: Quién es Dueño de Cada Libro, de Dónde Viene y si Está en Buen Estado 📚
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 04
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Big%20Data/04%20-%20Seguridad%20y%20Gobernanza%20de%20Big%20Data/Para%20Dummies/02_Gobernanza_Datos_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 02 del Módulo 04 (Gobernanza de datos)](../02_Gobernanza_de_Datos.ipynb). Imagina la **biblioteca de un pueblo de Boyacá** con miles de libros. Para que **no sea un caos** hace falta ponerse de acuerdo en **tres cosas**:

| En la biblioteca del pueblo | En gobernanza de datos |
|---|---|
| **La ficha de cada libro**: título, **quién lo donó** (el dueño), **quién lo cuida**, si es **solo para mayores** | **Catálogo de datos** con **propiedad** y **responsables** |
| **La receta de las arepas**: si se daña la leche, ¿qué productos se afectan? ¿de dónde sale el queso? | **Linaje de datos**: de dónde viene cada dato y qué depende de él |
| **La revisión de la lista de asistencia**: nombres vacíos, edades imposibles, teléfonos incompletos | **Reglas de calidad** y cuarentena |

La idea de fondo es simple: **datos sin dueño, sin ficha y sin revisar son datos en los que nadie responde**. *(Todos los datos de este cuaderno son **inventados**, y no usa nada más que Python básico y `pandas`.)* Al terminar podrás explicar:
1. Por qué **todo dato debe tener un dueño y un cuidador**, y cómo **encontrar los que no tienen**.
2. Cómo **seguir el camino** de un dato hacia atrás (de dónde viene) y hacia adelante (qué se rompe si cambia).
3. Cómo **revisar una tabla con reglas** sencillas y **separar lo dudoso** sin botarlo.

> 🔗 Si luego quieres ver la versión **con un catálogo completo, linaje dibujado como grafo, motor de reglas, políticas y contratos de datos**: [cuaderno estándar](../02_Gobernanza_de_Datos.ipynb). Antes de esto: [Candados, huellas y libros a prueba de borrones](01_Tecnicas_Seguridad_Dummies.ipynb).

---
## 1. La Ficha de Cada Libro: ¿de Quién es? 🏷️

Cada libro de la biblioteca tiene una **ficha** con su **dueño** (quien **responde** por él y decide quién lo puede leer), su **cuidador** (quien lo mantiene en buen estado) y si es **sensible** (por ejemplo, un registro con datos de los vecinos). Una **ficha incompleta** es un problema: si nadie es dueño, **nadie responde**. Revisamos el catálogo con cinco «libros» (inventados) y buscamos las **fichas con huecos**.

In [ ]:
import pandas as pd

catalogo = pd.DataFrame([
    ("registro_de_vecinos",  "Alcaldía (secretaría de gobierno)", "Equipo de archivo",   True),
    ("ventas_de_la_plaza",   "Asociación de comerciantes",         "Equipo de archivo",   False),
    ("lluvias_por_vereda",   "Observatorio del agua",              "Equipo del observatorio", False),
    ("lista_de_beneficiarios", "",                                 "Equipo de archivo",   True),      # NADIE es dueño
    ("encuesta_de_salud",    "Centro de salud",                    "",                    True),      # nadie la cuida
], columns=["libro", "dueño", "cuidador", "sensible"])

sin_dueno = catalogo[catalogo["dueño"] == ""].libro.tolist()
sin_cuidador = catalogo[catalogo["cuidador"] == ""].libro.tolist()
misma_persona = catalogo[(catalogo["dueño"] == catalogo["cuidador"]) & (catalogo["dueño"] != "")].libro.tolist()
print("Libros SIN dueño:", sin_dueno)
print("Libros SIN cuidador:", sin_cuidador)
print("Libros sensibles:", catalogo[catalogo.sensible].libro.tolist())
assert sin_dueno == ["lista_de_beneficiarios"] and sin_cuidador == ["encuesta_de_salud"] and misma_persona == []

sensibles_sin_responsable = catalogo[catalogo.sensible & ((catalogo["dueño"] == "") | (catalogo["cuidador"] == ""))].libro.tolist()
print("\n¡Lo más urgente! Datos sensibles con algún hueco en su ficha:", sensibles_sin_responsable)
assert sensibles_sin_responsable == ["lista_de_beneficiarios", "encuesta_de_salud"]

---
**Lección:** dos de los cinco «libros» tienen la ficha **incompleta** y **los dos son sensibles**: la lista de beneficiarios **no tiene dueño** (¿quién decide quién la puede ver?) y la encuesta de salud **no tiene cuidador** (¿quién corrige sus errores?). Eso es lo que la **gobernanza** busca evitar: que un dato delicado **exista sin que nadie responda por él**. Fíjate en la **diferencia entre los dos papeles**: el **dueño** *decide y responde* (es alguien del negocio, no del área de sistemas); el **cuidador** *mantiene en buen estado* (definiciones, calidad). Y el **área de sistemas** (los que guardan los archivos y los respaldos) **ejecuta**, pero **no es quien decide** quién entra. *(En el [cuaderno estándar](../02_Gobernanza_de_Datos.ipynb) el catálogo **se niega a registrar** una ficha incompleta, y hay una tabla de responsabilidades —quién hace, quién responde, a quién se consulta y a quién se informa— que se **revisa automáticamente**.)*

---

## 2. De la Vaca a la Arepa: el Camino de un Dato 🧀

En la plaza, la **arepa de queso** se hace con **queso** y **maíz**; el **queso** se hace con **leche**; el **pedido del colegio** lleva **arepas** y **jugo**; el **jugo** se hace con **mora**. Si **la leche llegó dañada**, ¿qué se afecta? Y si el pedido del colegio salió mal, ¿**de dónde vino** el problema? Responder eso es seguir el **linaje**: de qué se hizo cada cosa. Con los datos pasa **exactamente igual**: una tabla se calcula a partir de otras, y un tablero, a partir de esas.

In [ ]:
# "de que se hace cada cosa" (como una receta): producto -> ingredientes
receta = {"queso": ["leche"], "arepa de queso": ["queso", "maíz"], "jugo": ["mora"], "pedido del colegio": ["arepa de queso", "jugo"]}

def se_afecta(ingrediente, receta):
    """Todo lo que depende (directa o indirectamente) de `ingrediente`: hacia ADELANTE en la receta."""
    afectados = set()
    for producto, ingredientes in receta.items():
        if ingrediente in ingredientes:
            afectados |= {producto} | se_afecta(producto, receta)
    return afectados

def de_donde_viene(producto, receta):
    """Todos los ingredientes de los que depende `producto`: hacia ATRAS."""
    origen = set()
    for ing in receta.get(producto, []):
        origen |= {ing} | de_donde_viene(ing, receta)
    return origen

print("Si la leche llega dañada, se afecta:", sorted(se_afecta("leche", receta)))
print("El pedido del colegio viene de:", sorted(de_donde_viene("pedido del colegio", receta)))
assert se_afecta("leche", receta) == {"queso", "arepa de queso", "pedido del colegio"}
assert de_donde_viene("pedido del colegio", receta) == {"arepa de queso", "queso", "leche", "maíz", "jugo", "mora"}
assert se_afecta("mora", receta) == {"jugo", "pedido del colegio"} and se_afecta("pedido del colegio", receta) == set()

# ¿que ingrediente "pesa" mas, es decir, afecta a mas cosas si falla?
todos = {i for ings in receta.values() for i in ings} | set(receta)
peso = {x: len(se_afecta(x, receta)) for x in todos}
print("\n¿Cuántos productos se afectan si falla cada cosa?", dict(sorted(peso.items(), key=lambda kv: -kv[1])))
assert max(peso.values()) == 3

---
**Lección:** si la **leche** falla, se afectan **tres** cosas (el queso, la arepa y el pedido del colegio); si falla la **mora**, solo dos; y el pedido del colegio, que está **al final del camino**, no arrastra a nadie. Hacia **atrás**, el pedido viene de **seis** ingredientes. A esto se le llama **análisis de impacto** (¿a quién le toca si cambio esto?) y **búsqueda de la causa** (¿de dónde sale este resultado?). Con los datos es lo mismo: *«si cambio el nombre de una columna en esta tabla, ¿qué tableros se rompen?»*. Por eso, antes de tocar una fuente de datos, **se mira su linaje**. *(En el [cuaderno estándar](../02_Gobernanza_de_Datos.ipynb) el linaje es un **mapa dibujado** de 18 conjuntos de datos, y sirve también para saber que **algo hecho con datos reservados hereda su reserva**, salvo que se agrupe de forma anónima.)*

---

## 3. La Revisión de la Lista de Asistencia ✅

La profe de una escuela rural recibe la **lista de asistencia** (inventada) con **errores** de digitación: nombres vacíos, edades imposibles y teléfonos incompletos. Antes de usar la lista, la **revisa con reglas sencillas**; las filas que **no pasan** no se botan: se **apartan en una carpeta de dudosos** (cuarentena) para corregirlas después.

In [ ]:
lista = pd.DataFrame({
    "nombre":   ["Camila", "Andrés", "", "Luz Marina", "Jhon", "Paola", "", "Fabián"],
    "edad":     [8, 9, 7, 140, 10, 8, 9, -3],
    "telefono": ["3101234567", "3209876543", "3001112233", "31012345", "3157654321", "3114445566", "3201234567", "3109998877"],
})

reglas = {"nombre no vacío": lista.nombre.str.strip() != "",
          "edad entre 4 y 17": lista.edad.between(4, 17),
          "teléfono de 10 dígitos": lista.telefono.str.fullmatch(r"\d{10}")}

informe = pd.DataFrame({"regla": list(reglas), "filas que fallan": [int((~ok).sum()) for ok in reglas.values()]})
print(informe.to_string(index=False))

pasa_todo = pd.concat(reglas.values(), axis=1).all(axis=1)
buenas, dudosas = lista[pasa_todo], lista[~pasa_todo]
print(f"\nFilas buenas: {len(buenas)} | filas dudosas (a la carpeta de cuarentena): {len(dudosas)} | recibo: {len(buenas)} + {len(dudosas)} = {len(lista)}")
print("Dudosas:\n", dudosas.to_string())
assert informe["filas que fallan"].tolist() == [2, 2, 1] and len(buenas) + len(dudosas) == len(lista) == 8 and len(dudosas) == 4

---
**Lección:** las tres reglas marcaron **cinco fallos** (dos nombres vacíos, dos edades imposibles y un teléfono incompleto) que caen en **cuatro filas dudosas** de ocho (la de «Luz Marina» falla **dos** reglas a la vez: edad y teléfono); y lo importante: las dudosas **no se botaron**, quedaron **apartadas** y el **recibo cuadra** (4 buenas + 4 dudosas = 8). Botar en silencio sería como **romper hojas de la lista**: después nadie sabría que existieron. Y un **límite honesto**: las reglas solo cazan lo que **alguien pensó revisar**; una edad de **9** donde el niño tiene **10** pasa todas las revisiones y es un **error igual**.

---

## Resumen en una frase 🎯

> **Gobernar los datos es ponerles dueño y cuidador a todos (empezando por los sensibles), conocer de dónde viene cada uno y qué depende de él antes de cambiarlo, y revisarlos con reglas claras apartando lo dudoso en vez de botarlo.**

### Lo que aprendiste hoy:
- ✅ **Catálogo:** cada dato tiene ficha con **dueño** (decide y responde) y **cuidador** (mantiene); los sensibles sin responsable son lo más urgente.
- ✅ **Linaje:** seguir el camino **hacia adelante** (¿qué se afecta si falla?) y **hacia atrás** (¿de dónde viene?) antes de cambiar nada.
- ✅ **Calidad:** reglas sencillas separan lo bueno de lo dudoso; lo dudoso va a **cuarentena** y el **recibo cuadra**.

> 🎓 **Siguiente paso:** el [cuaderno estándar](../02_Gobernanza_de_Datos.ipynb) lo hace con un **catálogo de 18 conjuntos**, un **mapa de linaje**, un **motor de reglas**, **políticas escritas como código** y **contratos de datos**. Para seguir en el curso: **Módulo 05 · Avances y tendencias**.

---
##### 🛠️ Práctica 1: Agrega una regla: el nombre debe tener al menos 3 letras

Usa `lista`, `reglas` y `pasa_todo` de arriba:

1. Agrega a `reglas` una cuarta regla `"nombre de 3 letras o más"`: que `lista.nombre.str.strip().str.len() >= 3`.
2. Calcula `pasa_todo_2` (todas las reglas a la vez, con `pd.concat(reglas.values(), axis=1).all(axis=1)`) y `n_dudosas_2 = int((~pasa_todo_2).sum())`.
3. Comprueba con `assert` que `n_dudosas_2` es **igual** o **mayor** que las dudosas de antes (4): una regla nueva nunca deja pasar más filas. ¿Cambió el número? Mira cuántos nombres tienen menos de 3 letras y explícalo.

In [ ]:
# Escribe tu código aquí

# reglas["nombre de 3 letras o más"] = ...
# pasa_todo_2 = ...
# n_dudosas_2 = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
reglas["nombre de 3 letras o más"] = lista.nombre.str.strip().str.len() >= 3
pasa_todo_2 = pd.concat(reglas.values(), axis=1).all(axis=1)
n_dudosas_2 = int((~pasa_todo_2).sum())
print("Dudosas con la regla nueva:", n_dudosas_2, "(antes:", int((~pasa_todo).sum()), ")")
assert n_dudosas_2 >= int((~pasa_todo).sum())
assert n_dudosas_2 == 4
print("Sigue en 4: los nombres de menos de 3 letras son los mismos nombres vacíos que ya fallaban la primera regla. Una regla nueva solo ayuda si cubre algo que las otras no cubrían.")
```
</details>

---


---
### 🧠 ¿Ya lo tienes claro?

1. Explica con tus palabras la diferencia entre el **dueño**, el **cuidador** y el que **guarda los archivos**. ¿Quién decide si una persona nueva puede ver los datos de los vecinos?
2. Antes de **borrar** una tabla vieja, ¿qué mirarías en el «mapa de las recetas»? Da un ejemplo de lo que podría romperse sin que nadie lo notara.
3. Una fila de la lista de asistencia dice «edad 9» y el niño tiene 10 años. ¿La habrían detectado las reglas? ¿Qué se podría hacer (aparte de más reglas) para descubrir ese tipo de errores?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Big Data (Edición Didáctica Para Dummies)</i>
  </p>
</div>